In [1]:
import pandas as pd
import requests
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv()

# Project pathsx
DATA_RAW = Path("../data/raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)

print("Setup complete. Data will be saved to:", DATA_RAW.resolve())

Setup complete. Data will be saved to: /Users/ramyashreeb/Projects/explainable-ai-grid/data/raw


In [ ]:
# Quick test pull — current intensity
r = requests.get("https://api.carbonintensity.org.uk/intensity")
print(r.status_code)
print(r.json())

In [2]:
frames = []

for year in range(2021, 2024):
    for month in range(1, 13):
        url = f"https://api.carbonintensity.org.uk/intensity/{year}-{month:02d}-01T00:00Z/{year}-{month:02d}-28T23:30Z"
        r = requests.get(url)
        if r.status_code == 200:
            data = r.json().get("data", [])
            if data:
                frames.append(pd.DataFrame(data))
        else:
            print(f"Failed: {year}-{month:02d} (status {r.status_code})")

carbon_df = pd.concat(frames, ignore_index=True)
print(f"Carbon intensity: {carbon_df.shape}")
carbon_df.head()

Carbon intensity: (48237, 3)


,from,to,intensity
0,2020-12-31T23:30Z,2021-01-01T00:00Z,"{'forecast': 190, 'actual': 184, 'index': 'mod..."
1,2021-01-01T00:00Z,2021-01-01T00:30Z,"{'forecast': 181, 'actual': 187, 'index': 'mod..."
2,2021-01-01T00:30Z,2021-01-01T01:00Z,"{'forecast': 178, 'actual': 182, 'index': 'mod..."
3,2021-01-01T01:00Z,2021-01-01T01:30Z,"{'forecast': 175, 'actual': 178, 'index': 'mod..."
4,2021-01-01T01:30Z,2021-01-01T02:00Z,"{'forecast': 174, 'actual': 171, 'index': 'mod..."


In [3]:
frames = []

for year in range(2021, 2024):
    for month in range(1, 13):
        url = f"https://api.carbonintensity.org.uk/intensity/{year}-{month:02d}-01T00:00Z/{year}-{month:02d}-28T23:30Z"
        r = requests.get(url)
        if r.status_code == 200:
            data = r.json().get("data", [])
            if data:
                frames.append(pd.DataFrame(data))
        else:
            print(f"Failed: {year}-{month:02d} (status {r.status_code})")

carbon_df = pd.concat(frames, ignore_index=True)
print(f"Carbon intensity: {carbon_df.shape}")
carbon_df.head()

Carbon intensity: (48237, 3)


,from,to,intensity
0,2020-12-31T23:30Z,2021-01-01T00:00Z,"{'forecast': 190, 'actual': 184, 'index': 'mod..."
1,2021-01-01T00:00Z,2021-01-01T00:30Z,"{'forecast': 181, 'actual': 187, 'index': 'mod..."
2,2021-01-01T00:30Z,2021-01-01T01:00Z,"{'forecast': 178, 'actual': 182, 'index': 'mod..."
3,2021-01-01T01:00Z,2021-01-01T01:30Z,"{'forecast': 175, 'actual': 178, 'index': 'mod..."
4,2021-01-01T01:30Z,2021-01-01T02:00Z,"{'forecast': 174, 'actual': 171, 'index': 'mod..."


In [4]:
# Skip this cell until ENTSO_API_KEY is set in .env
ENTSO_KEY = os.getenv("ENTSO_API_KEY")

if ENTSO_KEY:
    from entsoe import EntsoePandasClient

    client = EntsoePandasClient(api_key=ENTSO_KEY)
    start = pd.Timestamp("2021-01-01", tz="Europe/London")
    end = pd.Timestamp("2024-01-01", tz="Europe/London")

    gen = client.query_generation("GB", start=start, end=end, psr_type=None)
    gen.to_parquet(DATA_RAW / "generation_GB_2021_2024.parquet")
    print(f"Generation: {gen.shape}")

    load = client.query_load("GB", start=start, end=end)
    load.to_parquet(DATA_RAW / "load_GB_2021_2024.parquet")
    print(f"Load: {load.shape}")
else:
    print("ENTSO_API_KEY not found in .env — register at transparency.entsoe.eu and add the key, then rerun this cell.")

ENTSO_API_KEY not found in .env — register at transparency.entsoe.eu and add the key, then rerun this cell.


In [5]:
carbon_df = pd.concat(
    [carbon_df.drop(columns=["intensity"]), carbon_df["intensity"].apply(pd.Series)],
    axis=1
)
carbon_df["from"] = pd.to_datetime(carbon_df["from"])
carbon_df["to"] = pd.to_datetime(carbon_df["to"])

carbon_df.to_parquet(DATA_RAW / "carbon_intensity_GB_2021_2024.parquet")
print("Saved:", DATA_RAW / "carbon_intensity_GB_2021_2024.parquet")
carbon_df.head()

Saved: ../data/raw/carbon_intensity_GB_2021_2024.parquet


,from,to,forecast,actual,index
0,2020-12-31 23:30:00+00:00,2021-01-01 00:00:00+00:00,190,184.0,moderate
1,2021-01-01 00:00:00+00:00,2021-01-01 00:30:00+00:00,181,187.0,moderate
2,2021-01-01 00:30:00+00:00,2021-01-01 01:00:00+00:00,178,182.0,moderate
3,2021-01-01 01:00:00+00:00,2021-01-01 01:30:00+00:00,175,178.0,moderate
4,2021-01-01 01:30:00+00:00,2021-01-01 02:00:00+00:00,174,171.0,moderate


In [6]:
cd ~/Projects/explainable-ai-grid
conda activate xai-energy
git add notebooks/01_data_ingestion.ipynb
git commit -m "feat: carbon intensity data pull complete, 48237 rows saved to parquet"
git push

SyntaxError: invalid decimal literal (3065828886.py, line 3)